<a href="https://colab.research.google.com/github/thekylebell/Momentum-Models/blob/v3.1/v3_vol_scaled_momentum_macro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Changes made from V3.0:

Faster momentum filter: added a short-term momentum signal (21 days) alongside the 252-day one, only going long when both agree, further reducing whipsaw trades

Transaction cost modeling: Every signal flip in v3.0 assumes frictionless trading, but even a 0.05% cost per trade will meaningfully impact results - especially on noisy signals

Walk-foward optimization: The volatility target was created as a static user input to test the feature in v3.0. Now, we find the optimal value on a rolling, in-sample window, rather than a fixed value.

Metrics export: Added user option which allows them to save results to a CSV for comparison without needing to re-run the program.

Sensitivity Analysis changes:

1 — Added fill_method=None to pct_change() to match the main model and avoid subtle return calculation errors

2 — Moved r_vol outside the inner loop since it's identical every iteration, removing redundant recalculation

3 — Added a per-vol progress print so the sweep doesn't look frozen during longer runs

4 — Red rectangle highlights the best-performing cell on every heatmap so the optimal parameters are immediately obvious

5 — Signal now requires both the 252-day and 21-day momentum to agree, keeping the heatmap consistent with what v3.1 actually trades

6 — Added a MAR ratio heatmap alongside Sharpe so drawdown risk is visible, not just return smoothness

7 — Transaction costs are now deducted from each simulated return, preventing the sweep from favouring high-turnover parameter combinations it can't actually achieve frictionlessly

8 — Data split 70/30 into in-sample and out-of-sample, with a full second set of heatmaps for OOS — parameters that hold up in both are genuinely robust, ones that collapse are overfit


In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas_datareader.data as web
from datetime import datetime

# --- 1. INTERACTIVE UI SETUP ---
ticker_input = widgets.Text(value='SPY', description='Asset:')
bench_input = widgets.Dropdown(
    options=[('S&P 500', 'SPY'), ('Gold', 'GLD'), ('Bitcoin', 'BTC-USD')],
    value='SPY', description='Bench:'
)
capital_input = widgets.FloatText(value=10000.0, description='Start $:', min=1.0, max=1e9)
vol_input = widgets.FloatSlider(value=0.12, min=0.05, max=0.40, step=0.01, description='Target Vol:')
cost_input = widgets.FloatText(value=0.0005, description='Cost/Trade:', min=0.0, max=0.01)
export_input = widgets.Checkbox(value=False, description='Export to CSV', indent=False)
start_date_input = widgets.DatePicker(value=pd.to_datetime('2015-01-01'), description='Start Date:')
run_button = widgets.Button(description="Execute Model", button_style='success')
output = widgets.Output()

ui = widgets.VBox([
    widgets.HBox([ticker_input, bench_input, capital_input]),
    widgets.HBox([vol_input, start_date_input, cost_input, export_input]),
    run_button
])

# --- 2. THE BACKTEST ENGINE ---
def run_backtest(b):
    global prices, tbill_daily
    with output:
        clear_output(wait=True)

        # Inputs
        ticker = ticker_input.value
        bench_ticker = bench_input.value
        start_cap = capital_input.value
        t_vol = vol_input.value
        start_str = start_date_input.value.strftime('%Y-%m-%d')

        print(f"Fetching data for {ticker}, {bench_ticker}, T-Bills, and CPI...")

        # A. Data Acquisition (Asset, Benchmark, 13-Week T-Bill)
        # ^IRX is the 13-week Treasury Bill yield
        data = yf.download([ticker, bench_ticker, '^IRX'], start=start_str, auto_adjust=False)

        # Handle yfinance MultiIndex
        if isinstance(data.columns, pd.MultiIndex):
            prices = data['Adj Close'].copy()
        else:
            prices = data[['Adj Close']].copy()

        prices = prices.dropna(subset=[ticker, '^IRX'])

        # B. Macro Data Acquisition (CPI from FRED)
        try:
            cpi = web.DataReader('CPIAUCSL', 'fred', start=start_str)
            cpi_daily = cpi.resample('D').ffill().pct_change(fill_method=None).fillna(0)
        except:
            print("Error fetching CPI. Defaulting to 3% annual estimate.")
            cpi_daily = pd.Series(0.03/252, index=prices.index)

        # C. Strategy Logic
        # 1. Dual Momentum Signal (252-day trend + 21-day confirmation)
        returns = prices[ticker].pct_change(fill_method=None)
        momentum_long  = (prices[ticker] / prices[ticker].shift(252)) - 1  # Trend filter
        momentum_short = (prices[ticker] / prices[ticker].shift(21))  - 1  # Whipsaw filter
        # Only go long when BOTH timeframes agree
        signal = ((momentum_long > 0) & (momentum_short > 0)).astype(int)

        # 2. Risk Engine (Volatility Scaling)
        realized_vol = returns.rolling(20).std() * np.sqrt(252)
        vol_weight = (t_vol / realized_vol).clip(upper=1.0)
        scaled_pos = (signal * vol_weight).shift(1)  # Lagging to prevent bias
        # 2. Risk Engine (Volatility Scaling)
        realized_vol = returns.rolling(20).std() * np.sqrt(252)
        vol_weight = (t_vol / realized_vol).clip(upper=1.0)
        scaled_pos = (signal * vol_weight).shift(1)  # Lagging to prevent bias

        # 3. Walk-Forward Vol Target Optimization (monthly rebalance)
        # For each month, look back 252 days and find the vol target
        # that maximized in-sample Sharpe. Apply it to the next 21 days.
        print("Running walk-forward optimization...")
        wf_lookback = 252
        vol_grid = np.arange(0.05, 0.41, 0.05)
        optimal_vol_series = pd.Series(t_vol, index=returns.index)

        rebal_indices = range(wf_lookback, len(returns), 21)
        for loc in rebal_indices:
            in_ret  = returns.iloc[loc - wf_lookback:loc]
            in_mom_long  = momentum_long.iloc[loc - wf_lookback:loc]
            in_mom_short = momentum_short.iloc[loc - wf_lookback:loc]
            in_sig  = ((in_mom_long > 0) & (in_mom_short > 0)).astype(int)
            in_rv   = in_ret.rolling(20).std() * np.sqrt(252)

            best_sharpe, best_vt = -np.inf, t_vol
            for vt in vol_grid:
                vw  = (vt / in_rv).clip(upper=1.0)
                pos = (in_sig * vw).shift(1)
                sr  = pos * in_ret
                sh  = (sr.mean() / sr.std() * np.sqrt(252)) if sr.std() != 0 else 0
                if sh > best_sharpe:
                    best_sharpe, best_vt = sh, vt

            next_loc = min(loc + 21, len(returns))
            optimal_vol_series.iloc[loc:next_loc] = best_vt

         # 4. Transaction Cost Drag
        # Each unit of position change costs cost_per_trade * position_delta
        trade_cost       = cost_input.value
        position_changes = scaled_pos.diff().abs().fillna(0)
        transaction_drag = position_changes * trade_cost
        total_trades     = (position_changes > 0).sum()

        # Override the static vol_weight with the walk-forward version
        vol_weight = (optimal_vol_series / realized_vol).clip(upper=1.0)
        scaled_pos = (signal * vol_weight).shift(1)

        # 3. Cash Management (T-Bills)
        # ^IRX is an annualized percentage. Convert to daily decimal.
        tbill_daily = (prices['^IRX'] / 100) / 252

        # D. Performance Math
        # Nominal Portfolio Return = (Position * Asset) + (Cash * T-Bill)
        nominal_strat_ret = (scaled_pos * returns) + ((1 - scaled_pos) * tbill_daily) - transaction_drag
        bench_ret = prices[bench_ticker].pct_change(fill_method=None)

        # Real Return Adjustment (Purchasing Power)
        # 1 + Real = (1 + Nominal) / (1 + Inflation)
        inf_daily = cpi_daily.reindex(nominal_strat_ret.index, method='ffill').iloc[:, 0]
        real_strat_ret = ((1 + nominal_strat_ret) / (1 + inf_daily)) - 1
        real_bench_ret = ((1 + bench_ret) / (1 + inf_daily)) - 1
        real_bench_wealth = start_cap * (1 + real_bench_ret.fillna(0)).cumprod()
        # E. Wealth Curves
        nominal_wealth = start_cap * (1 + nominal_strat_ret.fillna(0)).cumprod()
        real_wealth = start_cap * (1 + real_strat_ret.fillna(0)).cumprod()
        bench_wealth = start_cap * (1 + bench_ret.fillna(0)).cumprod()

        # --- CALCULATE DRAWDOWN ---
        # We use the Nominal Wealth curve for this
        rolling_max = nominal_wealth.cummax()
        drawdown = (nominal_wealth - rolling_max) / rolling_max
        max_drawdown = drawdown.min()
        # Benchmark Drawdown (for MAR comparison)
        bench_rolling_max = bench_wealth.cummax()
        bench_drawdown_curve = (bench_wealth - bench_rolling_max) / bench_rolling_max
        bench_max_drawdown = bench_drawdown_curve.min()

        # --- 3. VISUALIZATION ---
        # Create two subplots: one for growth, one for position size
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True, gridspec_kw={'height_ratios': [3, 1, 1]})

        # 1st Plot: Wealth
        ax1.plot(nominal_wealth, label='Strategy (Nominal)', color='teal')
        ax1.plot(real_wealth, label='Strategy (Real)', color='orange', linestyle='--')
        ax1.plot(bench_wealth, label=f'Benchmark ({bench_ticker})', color='gray', linestyle=':')
        ax1.plot(real_bench_wealth, label=f'Benchmark Real ({bench_ticker})', color='lightgray', linestyle=':')
        ax1.set_ylabel("Portfolio Value ($)")
        ax1.set_yscale('log')
        ax1.legend()

        # 2nd Plot: The "Dimmer Switch" (Position Size)
        ax2.fill_between(scaled_pos.index, 0, scaled_pos, color='teal', alpha=0.3, label='Market Exposure')
        ax2.set_ylabel("Exposure (0.0 to 1.0)")
        ax2.set_ylim(0, 1.1)
        ax2.legend()

        plt.suptitle(f"Strategy Analysis: {ticker}", fontsize=16)

        # Third Plot
        ax3.fill_between(drawdown.index, 0, drawdown * 100,
                         where=(drawdown >= -0.20), color='red', alpha=0.3, label='< 20% Drawdown')
        ax3.fill_between(drawdown.index, 0, drawdown * 100,
                         where=(drawdown < -0.20), color='darkred', alpha=0.7, label='> 20% Drawdown')
        ax3.axhline(y=-20, color='darkred', linestyle='--', linewidth=0.8, alpha=0.6)
        ax3.legend(fontsize=8)
        ax3.set_ylabel("Drawdown (%)")
        ax3.set_title("Portfolio Drawdown (Peak-to-Trough)", fontsize=10)
        ax3.grid(True, alpha=0.2)

        plt.tight_layout()
        plt.show()

        # --- 4. METRICS REPORT ---
        def get_sharpe(r): return (r.mean() / r.std()) * np.sqrt(252) if r.std() != 0 else 0

        total_nominal = nominal_wealth.iloc[-1]
        total_real = real_wealth.iloc[-1]

        # 1. Calculate CAGR (Annualized Return)
        total_days = (nominal_wealth.index[-1] - nominal_wealth.index[0]).days
        years = total_days / 365.25
        cagr = (nominal_wealth.iloc[-1] / start_cap)**(1/years) - 1
        bench_cagr = (bench_wealth.iloc[-1] / start_cap)**(1/years) - 1
        bench_mar = bench_cagr / abs(bench_max_drawdown) if bench_max_drawdown != 0 else 0
        # Calmar uses only the last 36 months, unlike MAR which uses full history
        calmar_wealth = nominal_wealth.loc[nominal_wealth.index >= nominal_wealth.index[-1] - pd.DateOffset(days=1096)] # ~36 months
        calmar_rolling_max = calmar_wealth.cummax()
        calmar_dd = ((calmar_wealth - calmar_rolling_max) / calmar_rolling_max).min()
        calmar_days = (calmar_wealth.index[-1] - calmar_wealth.index[0]).days
        calmar_cagr = (calmar_wealth.iloc[-1] / calmar_wealth.iloc[0])**(365.25/calmar_days) - 1
        calmar_ratio = calmar_cagr / abs(calmar_dd) if calmar_dd != 0 else 0
        # 2. Calculate MAR Ratio
        # We use abs() because Max Drawdown is a negative number
        mar_ratio = cagr / abs(max_drawdown) if max_drawdown != 0 else 0

        # 3. Print statements
        print(f"Strategy CAGR: {cagr*100:.2f}% | Benchmark CAGR: {bench_cagr*100:.2f}% | Aim is to beat benchmark")
        print(f"Strategy MAR: {mar_ratio:.2f} | Benchmark MAR: {bench_mar:.2f} | > 0.5: Good | > 1.0: Excellent | > 2.0: Holy Grail")
        print(f"Strategy Calmar Ratio (3yr): {calmar_ratio:.2f} | > 0.5: Good | > 1.0: Excellent | > 2.0: Holy Grail")
        print(f"Total Trades: {total_trades} | Total Cost Drag: {(transaction_drag.sum()*100):.3f}%")
        print("\n" + "="*30)
        print(f"FINAL PORTFOLIO VALUE: ${total_nominal:,.2f}")
        print(f"REAL PURCHASING POWER: ${total_real:,.2f}")
        print(f"LOSS TO INFLATION: ${(total_nominal - total_real):,.2f}")
        print("-"*30)
        print(f"Strategy Sharpe Ratio: {get_sharpe(nominal_strat_ret.dropna()):.2f} | > 1.0: Good | > 2.0: Very Good | > 3.0: Elite")
        print(f"Benchmark Sharpe Ratio: {get_sharpe(bench_ret.dropna()):.2f}")
        print("="*30)
        print(f"Max Strategy Drawdown: {max_drawdown*100:.2f}%")
        print('='*30)

        # --- 5. METRICS EXPORT ---
        if export_input.value:
            results_row = {
                'timestamp':        pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
                'ticker':           ticker,
                'benchmark':        bench_ticker,
                'start_date':       start_str,
                'start_capital':    start_cap,
                'target_vol':       t_vol,
                'cost_per_trade':   trade_cost,
                'total_trades':     total_trades,
                'cagr_pct':         round(cagr * 100, 2),
                'bench_cagr_pct':   round(bench_cagr * 100, 2),
                'mar_ratio':        round(mar_ratio, 2),
                'bench_mar':        round(bench_mar, 2),
                'calmar_ratio':     round(calmar_ratio, 2),
                'sharpe':           round(get_sharpe(nominal_strat_ret.dropna()), 2),
                'bench_sharpe':     round(get_sharpe(bench_ret.dropna()), 2),
                'max_drawdown_pct': round(max_drawdown * 100, 2),
                'final_nominal':    round(total_nominal, 2),
                'final_real':       round(total_real, 2),
            }

            export_path = 'backtest_results.csv'
            results_df  = pd.DataFrame([results_row])

            try:
                existing = pd.read_csv(export_path)
                results_df = pd.concat([existing, results_df], ignore_index=True)
            except FileNotFoundError:
                pass  # First run, no file yet

            results_df.to_csv(export_path, index=False)
            print(f"✅ Results saved to {export_path}")
        else:
            print("Export skipped. Check 'Export to CSV' to save results.")

run_button.on_click(run_backtest)
display(ui, output)

In [ ]:
import seaborn as sns

def run_heatmap_analysis():
    if 'prices' not in globals():
        print("Error: Please click 'Execute Model' in the first cell before running the heatmap.")
        return

    # 1. Define the ranges to test
    lookbacks = np.linspace(63, 378, 10).astype(int)
    vols      = np.linspace(0.05, 0.30, 10)

    # Use data already downloaded in the main cell
    asset_rets = prices[ticker_input.value].pct_change(fill_method=None)
    tbill      = (prices['^IRX'] / 100) / 252

    # Change 7: Pull transaction cost from the main cell's widget
    trade_cost = cost_input.value

    # Change 8: Out-of-sample split (70% in-sample, 30% out-of-sample)
    split_idx    = int(len(asset_rets) * 0.70)
    oos_asset    = asset_rets.iloc[split_idx:]
    oos_tbill    = tbill.iloc[split_idx:]
    oos_prices   = prices.iloc[split_idx:]
    is_asset     = asset_rets.iloc[:split_idx]
    is_tbill     = tbill.iloc[:split_idx]
    is_prices    = prices.iloc[:split_idx]

    # Change 2: Move r_vol outside the loop — it doesn't depend on v or l
    r_vol_full = asset_rets.rolling(20).std() * np.sqrt(252)
    r_vol_is   = is_asset.rolling(20).std() * np.sqrt(252)
    r_vol_oos  = oos_asset.rolling(20).std() * np.sqrt(252)

    # Grids to store Sharpe and MAR results (in-sample and out-of-sample)
    sharpe_is  = np.zeros((len(vols), len(lookbacks)))
    mar_is     = np.zeros((len(vols), len(lookbacks)))
    sharpe_oos = np.zeros((len(vols), len(lookbacks)))
    mar_oos    = np.zeros((len(vols), len(lookbacks)))

    print("Running parameter sweep...")

    for i, v in enumerate(vols):
        # Change 3: Progress indicator
        print(f"  Testing vol {v*100:.0f}%... ({i+1}/{len(vols)})", end='\r')
        for j, l in enumerate(lookbacks):

            # ----------------------------------------------------------------
            # IN-SAMPLE
            # Change 5: Dual momentum signal — long only when 252-day AND
            # 21-day signals agree, consistent with the main v3.1 model
            if l > 21:
                sig_long_is  = (is_prices[ticker_input.value] > is_prices[ticker_input.value].shift(l)).astype(int)
                sig_short_is = (is_prices[ticker_input.value] > is_prices[ticker_input.value].shift(21)).astype(int)
                sig_is       = ((sig_long_is == 1) & (sig_short_is == 1)).astype(int)
            else:
                sig_is = (is_prices[ticker_input.value] > is_prices[ticker_input.value].shift(l)).astype(int)

            weight_is    = (v / r_vol_is).clip(upper=1.0).shift(1)
            pos_is       = weight_is * sig_is

            # Change 7: Apply transaction cost drag
            drag_is      = pos_is.diff().abs().fillna(0) * trade_cost
            p_ret_is     = (pos_is * is_asset) + ((1 - pos_is) * is_tbill) - drag_is

            # Sharpe
            sharpe_is[i, j] = (p_ret_is.mean() / p_ret_is.std() * np.sqrt(252)) if p_ret_is.std() != 0 else 0

            # MAR (Change 6)
            wealth_is    = (1 + p_ret_is.fillna(0)).cumprod()
            roll_max_is  = wealth_is.cummax()
            max_dd_is    = ((wealth_is - roll_max_is) / roll_max_is).min()
            total_days_is = (wealth_is.index[-1] - wealth_is.index[0]).days
            if total_days_is > 0 and max_dd_is != 0:
                cagr_is      = (wealth_is.iloc[-1])**(365.25 / total_days_is) - 1
                mar_is[i, j] = cagr_is / abs(max_dd_is)

            # ----------------------------------------------------------------
            # OUT-OF-SAMPLE (Change 8)
            if l > 21:
                sig_long_oos  = (oos_prices[ticker_input.value] > oos_prices[ticker_input.value].shift(l)).astype(int)
                sig_short_oos = (oos_prices[ticker_input.value] > oos_prices[ticker_input.value].shift(21)).astype(int)
                sig_oos       = ((sig_long_oos == 1) & (sig_short_oos == 1)).astype(int)
            else:
                sig_oos = (oos_prices[ticker_input.value] > oos_prices[ticker_input.value].shift(l)).astype(int)

            weight_oos   = (v / r_vol_oos).clip(upper=1.0).shift(1)
            pos_oos      = weight_oos * sig_oos
            drag_oos     = pos_oos.diff().abs().fillna(0) * trade_cost
            p_ret_oos    = (pos_oos * oos_asset) + ((1 - pos_oos) * oos_tbill) - drag_oos

            sharpe_oos[i, j] = (p_ret_oos.mean() / p_ret_oos.std() * np.sqrt(252)) if p_ret_oos.std() != 0 else 0

            wealth_oos    = (1 + p_ret_oos.fillna(0)).cumprod()
            roll_max_oos  = wealth_oos.cummax()
            max_dd_oos    = ((wealth_oos - roll_max_oos) / roll_max_oos).min()
            total_days_oos = (wealth_oos.index[-1] - wealth_oos.index[0]).days
            if total_days_oos > 0 and max_dd_oos != 0:
                cagr_oos       = (wealth_oos.iloc[-1])**(365.25 / total_days_oos) - 1
                mar_oos[i, j]  = cagr_oos / abs(max_dd_oos)

    print(f"\nDone. Split: {prices.index[split_idx].date()} | In-sample: 70% | Out-of-sample: 30%")

    tick_labels_x = lookbacks
    tick_labels_y = [f"{v*100:.0f}%" for v in vols]

    # Change 6: Four heatmaps — Sharpe and MAR for both IS and OOS
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))

    datasets = [
        (sharpe_is,  axes[0, 0], "In-Sample Sharpe Ratio"),
        (mar_is,     axes[0, 1], "In-Sample MAR Ratio"),
        (sharpe_oos, axes[1, 0], "Out-of-Sample Sharpe Ratio"),
        (mar_oos,    axes[1, 1], "Out-of-Sample MAR Ratio"),
    ]

    for data, ax, title in datasets:
        sns.heatmap(data, annot=True, fmt=".2f",
                    xticklabels=tick_labels_x,
                    yticklabels=tick_labels_y,
                    cmap='viridis', ax=ax)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel("Momentum Lookback (Days)")
        ax.set_ylabel("Target Volatility (%)")

        # Change 4: Red box around the best cell
        best_i, best_j = np.unravel_index(np.argmax(data), data.shape)
        ax.add_patch(plt.Rectangle((best_j, best_i), 1, 1,
                                   fill=False, edgecolor='red', lw=2.5))

    plt.suptitle(
        f"Sensitivity Analysis: {ticker_input.value} — 70/30 In/Out-of-Sample Split\n"
        f"(Red box = best cell | Cost/trade: {trade_cost*100:.3f}%)",
        fontsize=14
    )
    plt.tight_layout()
    plt.show()

run_heatmap_analysis()